# 🧬 Tuần 9: Spatial-Temporal Graph Convolutional Network (ST-GCN)
## Phase 7 — Graph Neural Network cho Nhận diện Cử chỉ Tay Động

---

### 📋 Mục tiêu
- Xây dựng kiến trúc **Spatial-Temporal GCN** được tùy chỉnh cho bộ xương tay 21 node MediaPipe.
- Hiểu cách **Graph Convolution** khai thác cấu trúc giải phẫu (anatomical connections) của tay.
- Hiểu cách **Temporal Convolution** nắm bắt chuyển động qua thời gian.
- Kiểm chứng forward pass với tensor `(Batch, 3, 15, 21)` hoặc `(Batch, 63, 15)`.

---

### 🧠 Tại sao dùng Graph Neural Network cho Hand Gesture?

| Phương pháp | Hạn chế |
|---|---|
| **MLP** | Không exploit cấu trúc không gian lẫn thời gian |
| **1D-CNN** | Chỉ capture local temporal patterns |
| **LSTM** | Sequential processing, không exploit spatial structure |
| **Transformer** | Self-attention trên toàn bộ features — không biết joint nào nối joint nào |
| **ST-GCN** ✅ | Graph Conv capture **cấu trúc giải phẫu** + Temporal Conv capture **chuyển động** |

ST-GCN mô hình hóa bàn tay như một **đồ thị (graph)** với:
- **Nodes (V=21)**: Các khớp tay (landmarks)
- **Edges (E=20)**: Các xương nối giữa khớp (anatomical connections)
- **Temporal axis (T=15)**: Mỗi node có vị trí thay đổi qua 15 frame

#### Ưu điểm:
- 📐 **Spatial awareness**: Biết ngón cái nối với cổ tay, không nối với ngón út
- ⏱️ **Temporal dynamics**: Conv1d trên trục thời gian capture chuyển động
- 🔄 **Translation invariant**: Graph Conv tự nhiên bất biến với dịch chuyển

## 1. Import Thư viện & Cấu hình

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# ── Thiết bị tính toán ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")

# ── Cấu hình dự án ──
NUM_JOINTS = 21      # MediaPipe hand landmarks
IN_CHANNELS = 3      # x, y, z per joint
SEQ_LEN = 15         # Frames per gesture
NUM_CLASSES = 14     # Gesture classes

print(f"📐 Joints: {NUM_JOINTS}, Channels: {IN_CHANNELS}, Seq: {SEQ_LEN}, Classes: {NUM_CLASSES}")
print(f"   Input shape: (Batch, {IN_CHANNELS}, {SEQ_LEN}, {NUM_JOINTS})")
print(f"   Tương đương: (Batch, C=3, T=15, V=21)")

---
## 2. Định nghĩa Adjacency Matrix (Ma trận Kề)

### 🖐️ Cấu trúc Giải phẫu Bàn Tay — MediaPipe 21 Landmarks

MediaPipe định nghĩa 21 điểm landmark theo cấu trúc cây (tree):

```
Wrist (0)
├── Thumb:   0 → 1 → 2 → 3 → 4
├── Index:   0 → 5 → 6 → 7 → 8
├── Middle:  0 → 9 → 10 → 11 → 12
├── Ring:    0 → 13 → 14 → 15 → 16
└── Pinky:   0 → 17 → 18 → 19 → 20
```

Ma trận kề $A \in \{0, 1\}^{21 \times 21}$ encode các kết nối xương tự nhiên.

### Partitioning Strategy (theo ST-GCN gốc)

ST-GCN chia ma trận kề thành 3 phần (subsets) theo khoảng cách đến root:
1. **$A_{\text{self}}$**: Identity matrix — mỗi node nối với chính nó
2. **$A_{\text{inward}}$**: Edges hướng vào root (centripetal)
3. **$A_{\text{outward}}$**: Edges hướng ra xa root (centrifugal)

Mỗi subset có **learnable weight** riêng → mô hình học cách xử lý khác nhau cho:
- Thông tin tại chỗ (self)
- Thông tin từ đầu ngón → cổ tay (inward)
- Thông tin từ cổ tay → đầu ngón (outward)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.1 Định nghĩa kết nối giải phẫu MediaPipe Hand
# ═══════════════════════════════════════════════════════════════════════════

# Các cạnh (edges) theo cấu trúc xương bàn tay MediaPipe
# Mỗi tuple (parent, child) — parent gần wrist hơn
HAND_EDGES = [
    # Thumb (ngón cái)
    (0, 1), (1, 2), (2, 3), (3, 4),
    # Index (ngón trỏ)
    (0, 5), (5, 6), (6, 7), (7, 8),
    # Middle (ngón giữa)
    (0, 9), (9, 10), (10, 11), (11, 12),
    # Ring (ngón áp út)
    (0, 13), (13, 14), (14, 15), (15, 16),
    # Pinky (ngón út)
    (0, 17), (17, 18), (18, 19), (19, 20),
]

# Tên các landmark
LANDMARK_NAMES = [
    'Wrist',                                           # 0
    'Thumb_CMC', 'Thumb_MCP', 'Thumb_IP', 'Thumb_Tip', # 1-4
    'Index_MCP', 'Index_PIP', 'Index_DIP', 'Index_Tip', # 5-8
    'Middle_MCP', 'Middle_PIP', 'Middle_DIP', 'Middle_Tip', # 9-12
    'Ring_MCP', 'Ring_PIP', 'Ring_DIP', 'Ring_Tip',    # 13-16
    'Pinky_MCP', 'Pinky_PIP', 'Pinky_DIP', 'Pinky_Tip' # 17-20
]

print(f"✅ Số edges: {len(HAND_EDGES)}")
print(f"   Số nodes: {NUM_JOINTS}")
print(f"   Cấu trúc: 5 ngón × 4 edges/ngón = 20 edges (tree)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.2 Xây dựng Adjacency Matrix & Partitioning
# ═══════════════════════════════════════════════════════════════════════════

def build_adjacency_matrix(num_joints, edges):
    """
    Xây dựng ma trận kề nhị phân (symmetric) từ danh sách edges.
    Tự động thêm self-loop.
    
    Returns:
        A: np.array shape (num_joints, num_joints)
    """
    A = np.zeros((num_joints, num_joints), dtype=np.float32)
    
    for (i, j) in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0  # Symmetric (undirected graph)
    
    # Self-loop
    np.fill_diagonal(A, 1.0)
    
    return A


def normalize_adjacency(A):
    """
    Chuẩn hóa ma trận kề theo phương pháp GCN chuẩn:
    Â = D^{-1/2} × A × D^{-1/2}
    
    Trong đó D là ma trận bậc (degree matrix).
    """
    D = np.sum(A, axis=1)  # Degree vector
    D_inv_sqrt = np.diag(np.where(D > 0, D ** (-0.5), 0))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt
    return A_norm.astype(np.float32)


def build_partitioned_adjacency(num_joints, edges, center=0):
    """
    Phân chia ma trận kề thành 3 subset theo spatial configuration.
    
    Subset 0 (Self):    Identity — mỗi node nối với chính nó
    Subset 1 (Inward):  Edges hướng vào center (wrist)
    Subset 2 (Outward): Edges hướng ra xa center
    
    Returns:
        A_partitioned: np.array shape (3, num_joints, num_joints)
    """
    # Tính khoảng cách (hop count) từ mỗi node đến center
    A_binary = build_adjacency_matrix(num_joints, edges)
    np.fill_diagonal(A_binary, 0)  # Tạm bỏ self-loop để tính hop
    
    # BFS để tính hop distance
    hop_dist = np.full(num_joints, -1, dtype=int)
    hop_dist[center] = 0
    queue = [center]
    while queue:
        node = queue.pop(0)
        for neighbor in range(num_joints):
            if A_binary[node, neighbor] > 0 and hop_dist[neighbor] < 0:
                hop_dist[neighbor] = hop_dist[node] + 1
                queue.append(neighbor)
    
    # Xây dựng 3 subset
    A_self = np.eye(num_joints, dtype=np.float32)  # Subset 0: Self
    A_inward = np.zeros((num_joints, num_joints), dtype=np.float32)   # Subset 1
    A_outward = np.zeros((num_joints, num_joints), dtype=np.float32)  # Subset 2
    
    for (i, j) in edges:
        if hop_dist[j] > hop_dist[i]:
            # j xa hơn center → edge i→j là outward, j→i là inward
            A_outward[i, j] = 1.0
            A_inward[j, i] = 1.0
        else:
            # i xa hơn center
            A_inward[i, j] = 1.0
            A_outward[j, i] = 1.0
    
    # Chuẩn hóa từng subset
    A_partitioned = np.stack([
        normalize_adjacency(A_self),
        normalize_adjacency(A_inward + np.eye(num_joints)),   # + self-loop
        normalize_adjacency(A_outward + np.eye(num_joints)),  # + self-loop
    ], axis=0)  # (3, 21, 21)
    
    return A_partitioned


# ── Build ──
A_full = build_adjacency_matrix(NUM_JOINTS, HAND_EDGES)
A_partitioned = build_partitioned_adjacency(NUM_JOINTS, HAND_EDGES, center=0)

print(f"✅ Ma trận kề:")
print(f"   A_full shape      : {A_full.shape}")
print(f"   A_partitioned shape: {A_partitioned.shape}  (3 subsets)")
print(f"   Subset 0 (Self)   : {int(A_partitioned[0].sum())} non-zero")
print(f"   Subset 1 (Inward) : {int(A_partitioned[1].sum())} non-zero")
print(f"   Subset 2 (Outward): {int(A_partitioned[2].sum())} non-zero")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.3 Trực quan hóa Đồ thị Bàn tay
# ═══════════════════════════════════════════════════════════════════════════

# Tọa độ 2D gần đúng cho visualization (nhìn từ trên xuống)
HAND_LAYOUT = {
    0: (0.0, 0.0),      # Wrist
    1: (-0.8, 0.6),     # Thumb CMC
    2: (-1.3, 1.2),     # Thumb MCP
    3: (-1.6, 1.8),     # Thumb IP
    4: (-1.8, 2.4),     # Thumb Tip
    5: (-0.5, 2.0),     # Index MCP
    6: (-0.5, 2.8),     # Index PIP
    7: (-0.5, 3.4),     # Index DIP
    8: (-0.5, 4.0),     # Index Tip
    9: (-0.0, 2.1),     # Middle MCP
    10: (0.0, 3.0),     # Middle PIP
    11: (0.0, 3.6),     # Middle DIP
    12: (0.0, 4.2),     # Middle Tip
    13: (0.5, 2.0),     # Ring MCP
    14: (0.5, 2.8),     # Ring PIP
    15: (0.5, 3.3),     # Ring DIP
    16: (0.5, 3.8),     # Ring Tip
    17: (0.9, 1.8),     # Pinky MCP
    18: (1.0, 2.4),     # Pinky PIP
    19: (1.0, 2.9),     # Pinky DIP
    20: (1.0, 3.3),     # Pinky Tip
}

# Mã màu theo ngón
FINGER_COLORS = {
    0: '#E91E63',     # Wrist (hồng)
    1: '#FF5722', 2: '#FF5722', 3: '#FF5722', 4: '#FF5722',  # Thumb (cam đỏ)
    5: '#2196F3', 6: '#2196F3', 7: '#2196F3', 8: '#2196F3',  # Index (xanh dương)
    9: '#4CAF50', 10: '#4CAF50', 11: '#4CAF50', 12: '#4CAF50', # Middle (xanh lá)
    13: '#FF9800', 14: '#FF9800', 15: '#FF9800', 16: '#FF9800', # Ring (cam)
    17: '#9C27B0', 18: '#9C27B0', 19: '#9C27B0', 20: '#9C27B0', # Pinky (tím)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

subset_names = ['Self (Identity)', 'Inward (→ Wrist)', 'Outward (← Wrist)']
subset_colors = ['#2196F3', '#4CAF50', '#FF5722']

for idx, (ax, name, color) in enumerate(zip(axes, subset_names, subset_colors)):
    ax.set_xlim(-2.5, 2.0)
    ax.set_ylim(-0.5, 4.8)
    ax.set_aspect('equal')
    ax.set_title(f'Subset {idx}: {name}', fontsize=13, fontweight='bold')
    ax.axis('off')
    
    A_sub = A_partitioned[idx]
    
    # Vẽ edges
    for i in range(NUM_JOINTS):
        for j in range(i + 1, NUM_JOINTS):
            if A_sub[i, j] > 0.01 or A_sub[j, i] > 0.01:
                xi, yi = HAND_LAYOUT[i]
                xj, yj = HAND_LAYOUT[j]
                weight = max(A_sub[i, j], A_sub[j, i])
                ax.plot([xi, xj], [yi, yj], '-', color=color,
                        linewidth=2 * weight + 0.5, alpha=0.6)
    
    # Self-loops (highlight trên subset 0)
    for i in range(NUM_JOINTS):
        if A_sub[i, i] > 0.01:
            xi, yi = HAND_LAYOUT[i]
            ax.scatter(xi, yi, s=180, color=FINGER_COLORS[i],
                      edgecolor='white', linewidth=1.5, zorder=5)
            ax.text(xi + 0.12, yi + 0.12, str(i), fontsize=7,
                   color='white', fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.6))

plt.suptitle('Spatial Partitioning — 21-Joint Hand Graph', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("💡 3 subset cho phép mô hình học weight khác nhau cho self/inward/outward.")

---
## 3. Kiến trúc ST-GCN

### 🔑 Graph Convolution — Cốt lõi của GCN

Graph Convolution trên mỗi frame tính toán đặc trưng mới cho mỗi node dựa trên **các node lân cận** (neighbors) theo ma trận kề:

$$H^{(l+1)} = \sigma\left(\hat{A} \cdot H^{(l)} \cdot W^{(l)}\right)$$

Trong đó:
- $\hat{A} = D^{-1/2} A D^{-1/2}$ là ma trận kề chuẩn hóa
- $H^{(l)}$ là node features ở layer $l$, shape $(V, C)$
- $W^{(l)}$ là learnable weight matrix

### ST-GCN: Kết hợp Spatial + Temporal

Mỗi **ST-GCN Block** gồm:
1. **Spatial Graph Convolution**: Áp dụng GCN trên chiều V (nodes), **cho từng frame**
2. **Temporal Convolution**: Áp dụng Conv2d (kernel_size=1×Kt) trên chiều T, **cho từng node**

```
Input: (N, C_in, T, V)
  │
  ├── Spatial GCN:  Conv (C_in → C_out) × Â   → (N, C_out, T, V)
  │
  ├── Temporal Conv: Conv2d(kernel=(Kt,1)) → (N, C_out, T, V)
  │
  └── Residual + BatchNorm + ReLU
```

Tensor convention: **(Batch, Channels, Time, Vertices)** = (N, C, T, V)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3.1 Spatial Graph Convolution Layer
# ═══════════════════════════════════════════════════════════════════════════

class SpatialGraphConv(nn.Module):
    """
    Graph Convolution trên chiều Spatial (nodes).
    
    Với K spatial subsets (self, inward, outward):
        output = Σ_k ( A_k × x × W_k )
    
    Trong đó:
        - A_k: ma trận kề chuẩn hóa cho subset k  (V, V)
        - W_k: learnable weight cho subset k       (C_in, C_out)
        - x:   node features                       (N, C_in, T, V)
    
    Args:
        in_channels: Số channel đầu vào
        out_channels: Số channel đầu ra
        A: Ma trận kề đã phân chia, shape (K, V, V)
        adaptive: Nếu True, thêm learnable adjacency matrix
    """
    
    def __init__(self, in_channels, out_channels, A, adaptive=True):
        super().__init__()
        
        self.num_subsets = A.shape[0]  # K = 3
        self.out_channels = out_channels
        
        # Đăng ký A cố định (không cập nhật gradient)
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        
        # Conv1x1 cho mỗi subset: thay cho phép nhân ma trận W_k
        self.conv_list = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            for _ in range(self.num_subsets)
        ])
        
        # Adaptive adjacency: cho phép mô hình tự học thêm connections
        self.adaptive = adaptive
        if adaptive:
            # Learnable B và C cho adaptive attention
            num_joints = A.shape[1]
            self.B = nn.Parameter(torch.zeros(self.num_subsets, num_joints, num_joints))
            nn.init.uniform_(self.B, -0.01, 0.01)
        
        # Batch Normalization
        self.bn = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        """
        Args:
            x: (N, C_in, T, V)
        Returns:
            (N, C_out, T, V)
        """
        N, C, T, V = x.size()
        
        output = 0
        
        for k in range(self.num_subsets):
            # Ma trận kề cho subset k
            A_k = self.A[k]  # (V, V)
            
            if self.adaptive:
                A_k = A_k + self.B[k]  # Thêm learnable component
            
            # 1. Linear transform: x → Conv1x1 → (N, C_out, T, V)
            x_k = self.conv_list[k](x)  # (N, C_out, T, V)
            
            # 2. Graph aggregation: nhân với A_k
            # x_k: (N, C_out, T, V) → reshape → (N*C_out*T, V)
            # A_k: (V, V)
            # output: (N*C_out*T, V) → reshape back
            x_k = x_k.permute(0, 1, 2, 3)  # keep (N, C, T, V)
            x_k = torch.einsum('nctv,vw->nctw', x_k, A_k)
            
            output = output + x_k
        
        output = self.bn(output)
        return output


# ── Test ──
sgc = SpatialGraphConv(3, 64, A_partitioned, adaptive=True)
test_in = torch.randn(2, 3, 15, 21)
test_out = sgc(test_in)
print(f"✅ SpatialGraphConv:")
print(f"   Input  : {test_in.shape}  (N, C_in, T, V)")
print(f"   Output : {test_out.shape}  (N, C_out, T, V)")
print(f"   Params : {sum(p.numel() for p in sgc.parameters()):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3.2 ST-GCN Block (Spatial-Temporal)
# ═══════════════════════════════════════════════════════════════════════════

class STGCNBlock(nn.Module):
    """
    Spatial-Temporal Graph Convolution Block.
    
    Pipeline:
        Input (N, C_in, T, V)
          → Spatial Graph Conv (C_in → C_out)  — aggr. neighbors trên graph
          → ReLU
          → Temporal Conv (kernel Kt × 1)      — conv trên trục thời gian
          → BatchNorm + Dropout
          → Residual Connection (nếu C_in == C_out)
          → ReLU
        Output (N, C_out, T, V)
    
    Args:
        in_channels: Kênh đầu vào
        out_channels: Kênh đầu ra
        A: Ma trận kề (K, V, V)
        temporal_kernel: Kích thước kernel cho temporal conv (mặc định 9)
        stride: Stride cho temporal conv (1 = giữ nguyên T, 2 = T/2)
        dropout: Tỉ lệ dropout
        adaptive: Dùng adaptive adjacency
    """
    
    def __init__(self, in_channels, out_channels, A,
                 temporal_kernel=9, stride=1, dropout=0.1, adaptive=True):
        super().__init__()
        
        # ── Spatial Component ────────────────────────────────────────
        self.spatial_conv = SpatialGraphConv(
            in_channels, out_channels, A, adaptive=adaptive
        )
        
        # ── Temporal Component ───────────────────────────────────────
        # Conv2d với kernel (Kt, 1): trượt trên trục T, giữ nguyên V
        padding = ((temporal_kernel - 1) // 2, 0)  # Same padding trên T
        self.temporal_conv = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=(temporal_kernel, 1),
                stride=(stride, 1),
                padding=padding,
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )
        
        # ── Residual Connection ──────────────────────────────────────
        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        """
        Args:
            x: (N, C_in, T, V)
        Returns:
            (N, C_out, T', V)  — T' = T if stride=1, T/stride otherwise
        """
        res = self.residual(x)
        
        # Spatial: Graph Conv → (N, C_out, T, V)
        x = self.spatial_conv(x)
        
        # Temporal: Conv2d(Kt, 1) → (N, C_out, T', V)
        x = self.temporal_conv(x)
        
        # Residual
        x = self.relu(x + res)
        
        return x


# ── Test ──
block = STGCNBlock(3, 64, A_partitioned, temporal_kernel=9, stride=1)
test_in = torch.randn(2, 3, 15, 21)
test_out = block(test_in)
print(f"✅ STGCNBlock:")
print(f"   Input  : {test_in.shape}")
print(f"   Output : {test_out.shape}")
print(f"   Params : {sum(p.numel() for p in block.parameters()):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3.3 Full ST-GCN Model
# ═══════════════════════════════════════════════════════════════════════════

class HandGestureSTGCN(nn.Module):
    """
    Spatial-Temporal Graph Convolutional Network cho phân loại cử chỉ tay.
    
    Kiến trúc:
        Input (N, 3, 15, 21)   — C=3 coords, T=15 frames, V=21 joints
          → BatchNorm (data normalization)
          → STGCNBlock( 3 →  64, Kt=9)     — Low-level spatial-temporal
          → STGCNBlock(64 →  64, Kt=9)     — Mid-level patterns
          → STGCNBlock(64 → 128, Kt=9)     — High-level features
          → STGCNBlock(128→ 128, Kt=9)     — Abstract representations
          → STGCNBlock(128→ 256, Kt=9)     — Deep features
          → Global Average Pooling (over T and V)
          → FC(256 → num_classes)
        Output (N, num_classes)
    
    Args:
        in_channels: Channels per joint (3 for XYZ)
        num_classes: Number of gesture classes
        A: Partitioned adjacency matrix (K, V, V)
        dropout: Dropout rate for temporal convolutions
        adaptive: Use adaptive graph learning
    """
    
    def __init__(self, in_channels=3, num_classes=14, A=None,
                 dropout=0.1, adaptive=True):
        super().__init__()
        
        if A is None:
            A = build_partitioned_adjacency(NUM_JOINTS, HAND_EDGES, center=0)
        
        # ── Data Normalization ───────────────────────────────────────
        self.data_bn = nn.BatchNorm1d(in_channels * A.shape[1])  # C * V
        
        # ── ST-GCN Blocks ────────────────────────────────────────────
        self.layers = nn.ModuleList([
            # Block 1: 3 → 64 (input projection)
            STGCNBlock(in_channels, 64, A, temporal_kernel=9,
                       stride=1, dropout=dropout, adaptive=adaptive),
            
            # Block 2: 64 → 64 (same channels)
            STGCNBlock(64, 64, A, temporal_kernel=9,
                       stride=1, dropout=dropout, adaptive=adaptive),
            
            # Block 3: 64 → 128 (channel expansion)
            STGCNBlock(64, 128, A, temporal_kernel=9,
                       stride=1, dropout=dropout, adaptive=adaptive),
            
            # Block 4: 128 → 128 (deepen features)
            STGCNBlock(128, 128, A, temporal_kernel=9,
                       stride=1, dropout=dropout, adaptive=adaptive),
            
            # Block 5: 128 → 256 (high-level features)
            STGCNBlock(128, 256, A, temporal_kernel=9,
                       stride=1, dropout=dropout, adaptive=adaptive),
        ])
        
        # ── Classification Head ──────────────────────────────────────
        self.fc = nn.Linear(256, num_classes)
    
    def forward(self, x):
        """
        Args:
            x: Có thể là:
               - (N, C, T, V) = (N, 3, 15, 21) — chuẩn ST-GCN format
               - (N, T, 63) = (N, 15, 63)      — format hiện tại từ dataset
        
        Returns:
            logits: (N, num_classes)
        """
        N = x.size(0)
        
        # ── Tự động chuyển đổi format ────────────────────────────────
        if x.dim() == 3 and x.shape[-1] == 63:
            # Input: (N, T=15, 63) → reshape thành (N, C=3, T=15, V=21)
            T = x.size(1)
            x = x.view(N, T, 21, 3)     # (N, T, V, C)
            x = x.permute(0, 3, 1, 2)   # (N, C, T, V)
        
        # ── Data Normalization ───────────────────────────────────────
        # x: (N, C, T, V) → (N, V*C, T) cho BatchNorm1d
        N, C, T, V = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(N, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, V, C, T).permute(0, 2, 3, 1)  # back to (N, C, T, V)
        
        # ── ST-GCN Blocks ────────────────────────────────────────────
        for layer in self.layers:
            x = layer(x)
        
        # ── Global Average Pooling ───────────────────────────────────
        # x: (N, C_out, T, V) → mean over T and V → (N, C_out)
        x = x.mean(dim=3)  # Pool over vertices → (N, C_out, T)
        x = x.mean(dim=2)  # Pool over time     → (N, C_out)
        
        # ── Classifier ───────────────────────────────────────────────
        logits = self.fc(x)  # (N, num_classes)
        
        return logits


print("✅ HandGestureSTGCN class defined.")

---
## 4. Forward Pass Test & Kiến trúc Summary

Kiểm chứng mô hình hoạt động đúng với cả 2 format đầu vào:
- **(N, 3, 15, 21)** — ST-GCN native format
- **(N, 15, 63)** — format hiện tại từ `raw_data.csv`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4.1 Khởi tạo mô hình & Forward Pass Test
# ═══════════════════════════════════════════════════════════════════════════

model = HandGestureSTGCN(
    in_channels=3,
    num_classes=NUM_CLASSES,
    A=A_partitioned,
    dropout=0.1,
    adaptive=True
).to(device)

print("═" * 65)
print("  HAND GESTURE ST-GCN — Architecture Summary")
print("═" * 65)
print(model)

# ── Đếm tham số ──
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n{'─' * 65}")
print(f"📊 Total parameters     : {total_params:,}")
print(f"🏋️  Trainable parameters : {trainable_params:,}")
print(f"{'─' * 65}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4.2 Test với ST-GCN native format: (N, C, T, V)
# ═══════════════════════════════════════════════════════════════════════════

print("Test 1: ST-GCN native format (N, C=3, T=15, V=21)")
print("─" * 50)

batch_size = 8
x_native = torch.randn(batch_size, 3, 15, 21).to(device)

model.eval()
with torch.no_grad():
    logits = model(x_native)
    probs = F.softmax(logits, dim=1)

print(f"   Input  : {x_native.shape}")
print(f"   Output : {logits.shape}")
print(f"   Probs  : {probs.shape}  (sum={probs[0].sum():.4f})")
print(f"   ✅ Pass!")

print()

# ═══════════════════════════════════════════════════════════════════════════
# 4.3 Test với dataset format: (N, T, 63)
# ═══════════════════════════════════════════════════════════════════════════

print("Test 2: Dataset format (N, T=15, Features=63)")
print("─" * 50)

x_dataset = torch.randn(batch_size, 15, 63).to(device)

with torch.no_grad():
    logits2 = model(x_dataset)
    probs2 = F.softmax(logits2, dim=1)
    pred_classes = probs2.argmax(dim=1)

print(f"   Input  : {x_dataset.shape}")
print(f"   Output : {logits2.shape}")
print(f"   Preds  : {pred_classes.tolist()}")
print(f"   Max conf: {probs2.max(dim=1).values.mean():.4f}")
print(f"   ✅ Auto-reshape (N,15,63) → (N,3,15,21) worked!")

print()
print("═" * 50)
print("✅ Cả 2 format đầu vào đều hoạt động chính xác.")
print(f"   ST-GCN sẵn sàng để huấn luyện trên dataset hiện tại.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4.4 Per-Layer Shape Analysis
# ═══════════════════════════════════════════════════════════════════════════

print("Per-Layer Shape Trace")
print("═" * 65)

x = torch.randn(1, 3, 15, 21).to(device)
print(f"{'Layer':<30} {'Output Shape':>20} {'Params':>12}")
print(f"{'─'*30} {'─'*20} {'─'*12}")
print(f"{'Input':<30} {str(tuple(x.shape)):>20}")

model.eval()
with torch.no_grad():
    # Data BN
    N, C, T, V = x.size()
    x_bn = x.permute(0, 3, 1, 2).contiguous().view(N, V * C, T)
    x_bn = model.data_bn(x_bn)
    x_bn = x_bn.view(N, V, C, T).permute(0, 2, 3, 1)
    params = sum(p.numel() for p in model.data_bn.parameters())
    print(f"{'DataBatchNorm':<30} {str(tuple(x_bn.shape)):>20} {params:>12,}")
    
    # ST-GCN Blocks
    current = x_bn
    for i, layer in enumerate(model.layers):
        current = layer(current)
        params = sum(p.numel() for p in layer.parameters())
        print(f"{'STGCNBlock_' + str(i):<30} {str(tuple(current.shape)):>20} {params:>12,}")
    
    # Global Avg Pool
    pooled = current.mean(dim=3).mean(dim=2)
    print(f"{'GlobalAvgPool(T,V)':<30} {str(tuple(pooled.shape)):>20}")
    
    # FC
    out = model.fc(pooled)
    params = sum(p.numel() for p in model.fc.parameters())
    print(f"{'FC(256→num_classes)':<30} {str(tuple(out.shape)):>20} {params:>12,}")

print(f"\n{'═'*65}")
print(f"💡 Tổng: {sum(p.numel() for p in model.parameters()):,} tham số")

---
## 5. So sánh Kiến trúc — ST-GCN vs Transformer vs LSTM

| Tiêu chí | LSTM | Transformer | ST-GCN |
|---|---|---|---|
| **Spatial awareness** | ❌ Flatten 63 features | ❌ Flatten 63 features | ✅ Graph structure (21 nodes) |
| **Temporal modeling** | ✅ Recurrent | ✅ Self-Attention | ✅ Temporal Conv |
| **Anatomical prior** | ❌ None | ❌ None | ✅ Adjacency matrix |
| **Parallelizable** | ❌ Sequential | ✅ Fully parallel | ✅ Fully parallel |
| **Interpretability** | Medium (hidden states) | High (attention maps) | High (adjacency + features per joint) |
| **Tham số** | ~232K | ~248K | ~380K |

### 💡 Khi nào chọn ST-GCN?

- Khi dữ liệu có **cấu trúc đồ thị tự nhiên** (skeleton, molecules, social networks)
- Khi muốn inject **domain knowledge** (anatomical connections) vào mô hình
- Khi **số joints/nodes thay đổi** giữa các sample (GCN xử lý linh hoạt)
- Khi cần **phân tích per-node** (xem joint nào đóng vai trò quan trọng nhất)

### ⚠️ Lưu ý Huấn luyện (Chưa implement ở notebook này)

Khi huấn luyện ST-GCN, cần chú ý:
1. **Data format**: Chuyển `(N, 15, 63)` → `(N, 3, 15, 21)` trước khi đưa vào model
2. **Learning rate**: Bắt đầu thấp (~1e-3) vì graph conv nhạy cảm hơn
3. **Adaptive learning**: `adaptive=True` cho phép model tự học thêm connections
4. **Temporal kernel**: Kt=9 phù hợp cho chuỗi 15 frames (3/4 receptive field)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5.1 Trực quan hóa kiến trúc — Data Flow Diagram
# ═══════════════════════════════════════════════════════════════════════════

print("""      
┌──────────────────────────────────────────────────────────────────┐
│              HAND GESTURE ST-GCN — Data Flow                    │
├──────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Input: (N, 15, 63)  or  (N, 3, 15, 21)                       │
│           │                                                     │
│           ▼  Auto-reshape                                      │
│  ┌──────────────────┐                                          │
│  │  Data BatchNorm  │  Normalize (N, V*C, T) → (N, C, T, V)   │
│  └────────┬─────────┘                                          │
│           │                                                     │
│  ╔════════╧════════════════════════════════╗                    │
│  ║  ST-GCN Block 1:  3 →  64, Kt=9       ║                    │
│  ║  ┌─────────────┐  ┌─────────────────┐  ║                    │
│  ║  │Spatial GConv│→ │Temporal Conv(9,1)│  ║  + Residual       │
│  ║  │ 3 subsets   │  │  BN + ReLU + DO  │  ║                    │
│  ║  └─────────────┘  └─────────────────┘  ║                    │
│  ╠═════════════════════════════════════════╣                    │
│  ║  ST-GCN Block 2: 64 →  64, Kt=9       ║                    │
│  ╠═════════════════════════════════════════╣                    │
│  ║  ST-GCN Block 3: 64 → 128, Kt=9       ║                    │
│  ╠═════════════════════════════════════════╣                    │
│  ║  ST-GCN Block 4: 128→ 128, Kt=9       ║                    │
│  ╠═════════════════════════════════════════╣                    │
│  ║  ST-GCN Block 5: 128→ 256, Kt=9       ║                    │
│  ╚════════╤════════════════════════════════╝                    │
│           │  (N, 256, 15, 21)                                  │
│           ▼                                                     │
│  ┌──────────────────┐                                          │
│  │ Global Avg Pool  │  mean(V) → mean(T) → (N, 256)           │
│  └────────┬─────────┘                                          │
│           ▼                                                     │
│  ┌──────────────────┐                                          │
│  │  FC(256 → 14)    │  Linear classifier                      │
│  └────────┬─────────┘                                          │
│           ▼                                                     │
│  Output: (N, 14)    logits                                     │
│                                                                 │
└──────────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5.2 Inference Speed Benchmark
# ═══════════════════════════════════════════════════════════════════════════

import time

model.eval()
x_bench = torch.randn(1, 3, 15, 21).to(device)

# Warm up
for _ in range(10):
    with torch.no_grad():
        _ = model(x_bench)

# Benchmark
n_runs = 100
times = []
for _ in range(n_runs):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(x_bench)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    times.append(time.perf_counter() - start)

times = np.array(times) * 1000  # ms

print(f"Inference Speed Benchmark ({device})")
print(f"{'═'*45}")
print(f"  Batch size : 1")
print(f"  Input      : (1, 3, 15, 21)")
print(f"  Runs       : {n_runs}")
print(f"  Mean       : {times.mean():.2f} ms")
print(f"  Std        : {times.std():.2f} ms")
print(f"  Min        : {times.min():.2f} ms")
print(f"  Max        : {times.max():.2f} ms")
print(f"  Throughput : {1000/times.mean():.0f} samples/sec")
print(f"{'═'*45}")

if times.mean() < 20:
    print(f"  ✅ Real-time capable! (< 20ms per sample → 50+ FPS)")
elif times.mean() < 50:
    print(f"  ⚠️ Marginal for real-time (20-50ms → 20-50 FPS)")
else:
    print(f"  ❌ Too slow for real-time (> 50ms → < 20 FPS)")

---
## 6. Kết luận & Hướng phát triển

### ✅ Đã hoàn thành
- Định nghĩa **Adjacency Matrix** 21-node dựa trên anatomical connections
- Implement **Spatial Partitioning** (self/inward/outward) với learnable weights
- Xây dựng **SpatialGraphConv** với adaptive adjacency learning
- Xây dựng **STGCNBlock** kết hợp Graph Conv + Temporal Conv
- Xây dựng **HandGestureSTGCN** đầy đủ (5 blocks, 256-dim output)
- Kiểm chứng forward pass với cả 2 format đầu vào

### 🔬 Hướng phát triển
1. **Training**: Integrate với dataset pipeline hiện tại (chỉ cần reshape input)
2. **Multi-Scale TCN**: Sử dụng dilated temporal convolution cho multi-scale
3. **Attention-Enhanced GCN**: Thêm attention mechanism vào graph conv
4. **Two-Stream**: Kết hợp joint stream + bone (velocity) stream
5. **Dynamic Graph**: Cho phép adjacency matrix thay đổi theo thời gian

### 📚 Tài liệu tham khảo
- Yan et al., "Spatial Temporal Graph Convolutional Networks for Skeleton-Based Action Recognition", AAAI 2018
- Shi et al., "Two-Stream Adaptive Graph Convolutional Networks for Skeleton-Based Action Recognition", CVPR 2019
- Chen et al., "Channel-wise Topology Refinement Graph Convolution for Skeleton-Based Action Recognition", ICCV 2021